In [2]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, joblib
from sklearn.metrics import average_precision_score
from src.features import add_features, FEATURES

raw_tr = pd.read_csv("../data/raw/fraudTrain.csv", index_col=0, parse_dates=["trans_date_trans_time"])
raw_te = pd.read_csv("../data/raw/fraudTest.csv", index_col=0, parse_dates=["trans_date_trans_time"])
raw_tr["source"], raw_te["source"] = "train", "test"
full = add_features(pd.concat([raw_tr, raw_te], ignore_index=True))
test = full[full["source"] == "test"].copy()

for name in ["xgb_v1", "xgb_deploy"]:
    scores = joblib.load(f"../models/{name}.joblib").predict_proba(test[FEATURES])[:, 1]
    bins = pd.cut(scores, [0, 0.001, 0.01, 0.1, 0.5, 1.0], include_lowest=True)
    calib = (pd.DataFrame({"score": scores, "fraud": test["is_fraud"].to_numpy(), "bin": bins})
               .groupby("bin", observed=True)
               .agg(n=("fraud", "size"), mean_score=("score", "mean"), fraud_rate=("fraud", "mean")))
    print(f"\n{name} | PR-AUC {average_precision_score(test['is_fraud'], scores):.3f}")
    print(calib.round(4).to_string())


xgb_v1 | PR-AUC 0.962
                      n  mean_score  fraud_rate
bin                                            
(-0.001, 0.001]  542958      0.0000      0.0000
(0.001, 0.01]      7955      0.0030      0.0038
(0.01, 0.1]        2150      0.0312      0.0326
(0.1, 0.5]          657      0.2249      0.2131
(0.5, 1.0]         1999      0.9484      0.9465

xgb_deploy | PR-AUC 0.965
                      n  mean_score  fraud_rate
bin                                            
(-0.001, 0.001]  542372      0.0000      0.0000
(0.001, 0.01]      8448      0.0030      0.0028
(0.01, 0.1]        2272      0.0323      0.0312
(0.1, 0.5]          616      0.2293      0.2078
(0.5, 1.0]         2011      0.9489      0.9498
